Runs the complete SFT -> GRPO -> comparison pipeline for real headline results, as a Kaggle **background commit** rather than an interactive session -- `Save Version -> Save & Run All (Commit)` runs top to bottom on Kaggle's servers, independent of your browser tab, so it survives you closing the laptop. This matters more here than for a short run: GRPO's group sampling (`num_generations: 8`, colocated vLLM) makes this a multi-hour job, well past what a free Colab session reliably survives.

**Before running, in the notebook Settings panel (right sidebar):**
- **Accelerator**: GPU T4 (single or x2, either should work). A first attempt on this pipeline hung on both P100 and T4 -- not a GPU choice problem. The real cause: the install cell was missing `xformers`, which vLLM needs as a backup attention method on GPUs without FlashAttention-2 (true for every free Kaggle GPU). Fixed below.
- **Internet**: On (required to clone the repo, install packages, and download the base model / dataset).

**Session limits**: 12 hours per session, ~30 GPU-hours/week total. GRPO's `max_steps: 250` with 8 generations per prompt is the slow part; no end-to-end wall-clock estimate exists yet -- treat the first full attempt as the measurement, and record it in the README once it completes.

**Run the smoke test first** (small sample size, few GRPO steps) before a full run.

## 1. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Settings (right sidebar).'
print('GPU:', torch.cuda.get_device_name(0))
print('device count:', torch.cuda.device_count())
print('bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
import os

# FlashInfer JIT-compiles sampling kernels that link against libcuda.so.
# On Kaggle, that stub lives in /usr/local/cuda/lib64/stubs/ but is not
# on the default linker path -- the build fails with "cannot find -lcuda".
# Adding it here (before any install or import) fixes the link step.
_stubs = "/usr/local/cuda/lib64/stubs"
os.environ["LIBRARY_PATH"] = _stubs + ":" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = _stubs + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("CUDA stubs on linker path:", _stubs)

In [ ]:
import os
import subprocess

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # more context room

try:
    is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except Exception:
    is_t4 = False
vllm_pin = "vllm==0.11.2" if is_t4 else "vllm==0.15.1"
print("Installing", vllm_pin)

# Fix for a hang seen on both P100 and T4:
# - xformers and triton were missing
# - vLLM needs xformers as a backup attention method on GPUs without
#   FlashAttention-2 (true for every free Kaggle GPU)
# - without it, vLLM stalls silently instead of showing an error
# Matches Unsloth's own Kaggle GRPO notebook install steps.
!pip install -q --upgrade uv
!uv pip install -q --system --upgrade {vllm_pin} torchvision bitsandbytes xformers triton unsloth unsloth_zoo mlflow
!uv pip install -q --system transformers==4.56.2
!uv pip install -q --system --no-deps trl==0.22.2

In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Hydaspex/grpo-math-reasoning.git'
BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/grpo-math-reasoning')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone --branch $BRANCH $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
!pip install -q --no-deps -e .

src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from mathrl.config import load_config
print('mathrl imported successfully from', Path.cwd())

## 4. Configure the full run

The committed config already runs the full GSM8K split (`data.max_samples: null`), unlike the Colab notebook's deliberate smoke-run override. Two overrides are still needed: an absolute MLflow store path (survives being packaged into the commit's Output), and checkpoint `output_dir`s moved **outside** `REPO_DIR` -- cell 3 deletes `REPO_DIR` on every re-run, so leaving checkpoints inside it means a routine re-run silently wipes training progress.

In [ ]:
import yaml

config_path = Path('configs/grpo_qwen25_1_5b.yaml')
config = yaml.safe_load(config_path.read_text())

MLFLOW_DB = (REPO_DIR / 'mlflow.db').resolve()
config.setdefault('mlflow', {})['tracking_uri'] = f'sqlite:////{MLFLOW_DB}'

# Checkpoints must live outside REPO_DIR: the clone cell rmtree's REPO_DIR
# on every re-run, which would otherwise delete them too.
CHECKPOINT_ROOT = Path('/kaggle/working/outputs')
config['sft']['output_dir'] = str(CHECKPOINT_ROOT / 'qwen25-gsm8k-sft')
config['grpo']['output_dir'] = str(CHECKPOINT_ROOT / 'qwen25-gsm8k-grpo')

config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

## 4b. Smoke test first

Run the pipeline small before running it full. This checks that every
step works end to end in minutes, not hours -- cheap to redo if
something is still broken.

Set `SMOKE_TEST = False` only after a smoke run has passed.

In [ ]:
SMOKE_TEST = True

if SMOKE_TEST:
    config['data']['max_samples'] = 50
    config['grpo']['max_steps'] = 10
    config['grpo']['num_generations'] = 4
    config['sft']['max_steps'] = 10

config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('SMOKE_TEST =', SMOKE_TEST)
print(config_path.read_text())

## 5. Prepare GSM8K data

Writes prompt-only records for GRPO, supervised records for the control arm, and a shared validation split -- all from the same examples.

In [ ]:
!python scripts/prepare_data.py --config configs/grpo_qwen25_1_5b.yaml

## 6. SFT control arm

Trains on the same examples, targeting the same `<reasoning>`/`<answer>` format the GRPO reward pays out for. Without this arm, a GRPO improvement can't be distinguished from "any fine-tuning on this format would have helped."

In [ ]:
!python scripts/train_sft.py --config configs/grpo_qwen25_1_5b.yaml

## 7. GRPO training

**Reward will sit near zero and barely move for the first 100-200 steps -- that is expected, not a failure.** Correctness fires rarely early on; the format rewards carry the signal until it does. This is also the long step: 250 steps x 8 generations per prompt through a colocated vLLM engine.

In [ ]:
!python scripts/train_grpo.py --config configs/grpo_qwen25_1_5b.yaml

## 8. Check artefacts

In [ ]:
for path in [Path(config['sft']['output_dir']), Path(config['grpo']['output_dir'])]:
    print(path, 'exists:', path.exists())
    if path.exists():
        for item in list(path.iterdir())[:10]:
            print('  ', item.name)

## 9. Back up checkpoints to a Kaggle Dataset

`/kaggle/working` only survives as this notebook version's committed Output -- it is not a live store the *next* session or commit resumes from. To actually reuse checkpoints across sessions, snapshot them into a Kaggle Dataset: durable, versioned, and available as an input to a future run. Safe to skip if this run finished in one go. Set `KAGGLE_USERNAME` before running -- this cell is optional.

In [ ]:
import json

# Set this to your Kaggle username (shown in kaggle.com/<username>) before running.
KAGGLE_USERNAME = 'your-kaggle-username'
DATASET_SLUG = 'grpo-math-reasoning-checkpoints'
STAGING_DIR = Path('/kaggle/working/checkpoint_dataset')
STAGING_DIR.mkdir(exist_ok=True)

for src in [Path(config['sft']['output_dir']), Path(config['grpo']['output_dir'])]:
    if src.exists():
        dest = STAGING_DIR / src.name
        shutil.copytree(src, dest, dirs_exist_ok=True)

metadata = {
    'title': 'GRPO math-reasoning checkpoints',
    'id': f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(STAGING_DIR / 'dataset-metadata.json').write_text(json.dumps(metadata))

print(f'Staged checkpoints at {STAGING_DIR}. Create the dataset once (needs Kaggle API credentials):')
print(f'  kaggle datasets create -p {STAGING_DIR}')
print('Then on later versions, update it with:')
print(f'  kaggle datasets version -p {STAGING_DIR} -m "checkpoint update"')

## 10. Compare base, SFT and GRPO on the full validation split

This is the headline-numbers step: batched generation, full split, bootstrap confidence intervals, and a one-sided paired McNemar test between adjacent arms -- the test that answers "is GRPO actually better than SFT", not just "do the intervals overlap."

In [ ]:
!python scripts/compare_models.py \
    --config configs/grpo_qwen25_1_5b.yaml \
    --sft-adapter {config['sft']['output_dir']} \
    --grpo-adapter {config['grpo']['output_dir']} \
    --batch-size 8

## 11. Review runs in MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])

experiments = mlflow.search_experiments()
print(f'Found {len(experiments)} experiments.')

runs = mlflow.search_runs(experiment_ids=[e.experiment_id for e in experiments])
wanted = [
    'tags.mlflow.runName', 'metrics.accuracy', 'metrics.accuracy_ci_low',
    'metrics.accuracy_ci_high', 'metrics.format_valid',
]

if runs.empty:
    print(f'No runs found at {mlflow.get_tracking_uri()}.')
else:
    present = [c for c in wanted if c in runs.columns]
    if 'metrics.accuracy' in present:
        table = runs[present].rename(columns={'tags.mlflow.runName': 'run_name'})
        print(table.dropna(subset=['metrics.accuracy']).sort_values('metrics.accuracy', ascending=False))
    else:
        print('Runs found but missing metric columns. Columns:', runs.columns.tolist())

    mcnemar_cols = [c for c in runs.columns if c.startswith('metrics.mcnemar_p_')]
    if mcnemar_cols:
        comparison_run = runs[runs['tags.mlflow.runName'].str.contains('comparison', na=False)]
        if not comparison_run.empty:
            print('\nMcNemar p-values (comparison run):')
            print(comparison_run[mcnemar_cols].iloc[0])

## After this run

Record the actual GPU type, wall-clock time and peak VRAM alongside the results -- the README's Results section should cite these, not just the numbers, so the experiment is reproducible. This is also the first real run of this pipeline: note the actual end-to-end duration here, since no prior estimate exists to compare against.

To run this unattended: `Save Version -> Save & Run All (Commit)` from the notebook menu. Progress and final output are visible under the notebook's Output tab once it completes; the MLflow sqlite store (`mlflow.db`) and prediction JSONL files are included in that Output.